# 🚀 B3 Data Lake - Extração Máxima (YFinance)

Este notebook implementa a extração de dados da B3 em 4 níveis de profundidade, criando uma base completa para análise de dados e machine learning.

**Níveis de Extração:**
1. **Preços Históricos (OHLCV):** 10 anos de histórico diário.
2. **Eventos Corporativos:** Dividendos e Stock Splits (Desdobramentos).
3. **Metadados (Info):** Indicadores fundamentalistas e dados cadastrais (+100 colunas).
4. **Balanços (Financials):** Demonstração de Resultado (DRE) anual.

In [ ]:
import yfinance as yf
import pandas as pd
import fundamentus
from pathlib import Path
from datetime import datetime, timedelta
import warnings
import logging

# Configurações
warnings.filterwarnings('ignore')
logging.getLogger('yfinance').setLevel(logging.CRITICAL)

# Caminho para salvar os dados (Relativo à pasta notebooks)
PASTA_RAW = Path("../data/raw")
PASTA_RAW.mkdir(parents=True, exist_ok=True)

print("✅ Ambiente configurado e pastas verificadas.")

## 0. Mapeamento do Universo B3
Descobrimos todos os tickers ativos usando a biblioteca `fundamentus`.

In [ ]:
def obter_todos_tickers_b3():
    print("🌍 A mapear todo o universo de ações da B3...")
    try:
        df_ativos = fundamentus.get_resultado()
        tickers = [str(ticker) + ".SA" for ticker in df_ativos.index.tolist()]
        print(f"   🎯 Sucesso! Encontradas {len(tickers)} empresas ativas na B3.")
        return tickers
    except Exception as e:
        print(f"   ❌ Erro ao buscar lista de ativos: {e}")
        return ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "BBAS3.SA", "WEGE3.SA"]

tickers_b3 = obter_todos_tickers_b3()

## 1. Nível 1: Preços Históricos (OHLCV)
Extração massiva usando processamento paralelo (Multithreading).

In [ ]:
def extrair_precos_maciamente(tickers, data_inicio):
    print(f"📉 Download massivo de PREÇOS Históricos (10 anos)...")
    try:
        # Download paralelo
        dados = yf.download(tickers, start=data_inicio, threads=True, ignore_tz=True)
        
        # Transforma o MultiIndex em um DataFrame plano plano
        df = dados.stack(level=1).reset_index()
        df.rename(columns={'level_1': 'Ticker'}, inplace=True)
        
        print(f"   ✅ Preços extraídos! {len(df)} linhas salvas.")
        return df
    except Exception as e:
        print(f"   ❌ Erro: {e}")
        return pd.DataFrame()

hoje = datetime.today()
data_10_anos = (hoje - timedelta(days=365*10)).strftime('%Y-%m-%d')

df_precos = extrair_precos_maciamente(tickers_b3, data_10_anos)
df_precos.to_csv(PASTA_RAW / "01_yfinance_precos_raw.csv", sep=';', decimal=',', index=False, encoding='utf-8-sig')

## 2. Nível 2: Eventos Corporativos
Coleta de Dividendos e Splits.

In [ ]:
def extrair_eventos_corporativos(tickers):
    print(f"🔄 A extrair EVENTOS CORPORATIVOS de {len(tickers)} ações...")
    df_eventos = pd.DataFrame()
    
    for i, ticker in enumerate(tickers):
        if i % 100 == 0: print(f"   Progresso: {i}/{len(tickers)}")
        try:
            acao = yf.Ticker(ticker)
            acoes_corp = acao.actions
            if not acoes_corp.empty:
                df_temp = acoes_corp.reset_index()
                df_temp['Ticker'] = ticker.replace('.SA', '')
                df_eventos = pd.concat([df_eventos, df_temp], ignore_index=True)
        except: pass
            
    if not df_eventos.empty and 'Date' in df_eventos.columns:
        df_eventos['Date'] = pd.to_datetime(df_eventos['Date'], utc=True).dt.tz_localize(None)
        
    return df_eventos

df_eventos = extrair_eventos_corporativos(tickers_b3)
df_eventos.to_csv(PASTA_RAW / "02_yfinance_eventos_raw.csv", sep=';', decimal=',', index=False, encoding='utf-8-sig')

## 3. Nível 3: Metadados e Fundamentos (.info)
Captura de Beta, Per, Setor, Industry e mais.

In [ ]:
def extrair_info_avancada(tickers):
    print(f"🧠 A extrair METADADOS E FUNDAMENTOS GLOBAIS...")
    lista_infos = []
    
    for i, ticker in enumerate(tickers):
        if i % 100 == 0: print(f"   Progresso: {i}/{len(tickers)}")
        try:
            info = yf.Ticker(ticker).info
            if info and 'symbol' in info:
                info['Ticker'] = ticker.replace('.SA', '')
                lista_infos.append(info)
        except: pass
            
    return pd.DataFrame(lista_infos)

df_info = extrair_info_avancada(tickers_b3)
df_info.to_csv(PASTA_RAW / "03_yfinance_info_raw.csv", sep=';', decimal=',', index=False, encoding='utf-8-sig')

## 4. Nível 4: Balanços Financeiros
DREs anuais e trimestrais padronizadas.

In [ ]:
def extrair_balancos_anuais(tickers):
    print(f"📊 A extrair BALANÇOS FINANCEIROS (Income Statement)...")
    df_balancos = pd.DataFrame()
    
    for i, ticker in enumerate(tickers):
        if i % 100 == 0: print(f"   Progresso: {i}/{len(tickers)}")
        try:
            dre = yf.Ticker(ticker).financials 
            if not dre.empty:
                df = dre.T.reset_index().rename(columns={'index': 'Data_Balanco'})
                df['Ticker'] = ticker.replace('.SA', '')
                df_balancos = pd.concat([df_balancos, df], ignore_index=True)
        except: pass
            
    return df_balancos

df_balancos = extrair_balancos_anuais(tickers_b3)
df_balancos.to_csv(PASTA_RAW / "04_yfinance_balancos_raw.csv", sep=';', decimal=',', index=False, encoding='utf-8-sig')

### ✅ Finalizado
Os arquivos CSV foram gerados na pasta `data/raw`.